# Stable Diffusion Ecosystem- From Understanding to Shipping

**Module 6 · Lesson 6.2**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Inside the Pipeline - Three Parts Assembled
- Text-to-Image and the Control Dials
- Image-to-Image and Inpainting - Editing, Not Just Creating
- The Model Zoo - SD1.5 to Flux.2
- Customize It - LoRA and Fine-Tuning
- Ship It - Tooling, Speed, and Production

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# torch is preinstalled in Colab; diffusers/transformers/accelerate are not:
!pip install diffusers transformers accelerate -q

# Reproducibility - the lesson uses seeded random values throughout

## The whole game: one open pipeline you can take apart and rebuild

> 🎬 **Analogy**
>
> **A closed image API is a streaming subscription; Stable Diffusion is the film studio itself.** With DALL-E or Midjourney you watch what they allow - a prompt in, an image out, no peeking inside. Stable Diffusion hands you the whole studio: the writer (text encoder), the director (denoiser), the lab (VAE), plus a marketplace of lenses and effects (LoRAs, checkpoints) you can swap and even build yourself, all running on your own GPU.
> That openness is the entire lesson. Because the pipeline comes apart, **you control every stage** - the model, the dials, the edits, the style - instead of accepting a black box. The skill here is knowing which part to reach for.
> **Where the analogy breaks down:** a film studio's departments are loosely coupled, but the Stable Diffusion parts are trained tightly to each other. You cannot drop an SD1.5 lab (VAE) into a Flux production and expect it to work - the parts are swappable *within* a model family, not across families. The mounts have to match.

We build on Lesson 6.1's diffusion foundations - the denoiser that predicts noise, latent space, the noise schedule, and classifier-free guidance - and turn them into a working, shippable system with the `diffusers` library. Two big pieces are deliberately held back: structural control (ControlNet) gets its own treatment in Lesson 6.3, and the CLIP text encoder's inner workings are Lesson 6.4. Here, the focus is the ecosystem you operate.

- **Explain** how the three Stable Diffusion components - text encoder, denoiser, and VAE - assemble into a pipeline and map back to the diffusion process from Lesson 6.1.

- **Apply** the `diffusers` library to generate and edit images - text-to-image, image-to-image, and inpainting - and tune the control dials (steps, guidance, sampler, seed, negative prompt).

- **Compare** the model zoo (SD1.5, SDXL, SD3.5, Flux.1/Flux.2) and customization options (LoRA, DreamBooth, full fine-tune) to choose the right one for a task, budget, and license.

- **Analyze** a production image pipeline for cost, speed, licensing, and safety, and select tooling (ComfyUI, few-step models, serverless GPU) accordingly.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against Lesson 6.1. Each one is load-bearing for today.

---

## 🎯 Concept 1: Inside the Pipeline - Three Parts Assembled

### Inside the Pipeline - Three Parts Assembled

Stable Diffusion is not one model. It is three trained components plus a scheduler, working together.

**A three-person film crew.** A writer reads your idea and writes the brief; a director builds the scene step by step from the brief; a cinematographer handles the camera, shooting small and blowing it up. Hand the same idea to a different director and you get a different film.

The gain: text encoder = writer, denoiser = director, VAE = camera. Three parts, each replaceable - that is what makes it an ecosystem.

**📝 `explore_components.py`** - *diffusers*

In [ ]:
# pip install diffusers transformers accelerate torch
from diffusers import StableDiffusionXLPipeline
import torch

# A "checkpoint" bundles all three components into one pipeline object.
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,   # 16-bit half precision (~half the GPU memory)
)

# Peek inside - the three separately trained parts:
def count(m):
    return sum(p.numel() for p in m.parameters())

print(f"text encoder(s): {count(pipe.text_encoder) + count(pipe.text_encoder_2):,} params")
print(f"denoiser (UNet): {count(pipe.unet):,} params")
print(f"VAE:             {count(pipe.vae):,} params")
print(f"scheduler:       {type(pipe.scheduler).__name__}")
# Output: text encoder(s): 817,785,409 params   (SDXL has TWO CLIP encoders)
# Output: denoiser (UNet): 2,567,463,684 params  (the big one)
# Output: VAE:             83,653,863 params
# Output: scheduler:       EulerDiscreteScheduler

- `from_pretrained` downloads and assembles all three components plus a scheduler into one `pipe` object - that is what a "checkpoint" is.

- **The text encoder** (SDXL runs two CLIP encoders - a bigger and a smaller one - for richer prompt understanding) turns your prompt into guidance vectors. It does not draw - it tells the denoiser what to aim for. Its inner workings are Lesson 6.4.

- **The denoiser** (`unet`) is the largest part and the one from 6.1: predict noise, remove a slice, repeat - now guided by the text vectors. (A "parameter" is one learned number and `p.numel()` counts them, so the denoiser's ~2.5B dwarfing the VAE's ~84M just means it is the biggest sub-network.) This is where LoRAs and ControlNet attach.

- **The VAE** compresses images to latents (8x smaller per side, e.g. 512x512 -> 64x64) and decodes the final latent back to pixels. The `scheduler` runs the denoising loop - note that **diffusers calls the sampler a "scheduler"**, so those two words mean the same thing (it shows up as the sampler dial in Step 2). Swap any one component and the output changes - that swappability is the ecosystem.

- **Setup tokens** - `torch_dtype=torch.float16` loads the model in 16-bit half precision (about half the GPU memory, standard for SDXL); the snippets below add `.to("cuda")` to put the model on an NVIDIA GPU - use `.to("cpu")` if you have no CUDA GPU, though it is far slower.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  A Stable Diffusion checkpoint is really **three separately trained networks plus a scheduler**: a text encoder that turns words into guidance vectors, the denoiser from 6.1 that builds the image in latent space, and a VAE that compresses and then decodes to pixels. Because the parts are separable, you can swap and customize each one - which is the foundation for everything else in this lesson.

---

## 🎯 Concept 2: Text-to-Image and the Control Dials

### Text-to-Image and the Control Dials

Five lines to generate - then the handful of dials that decide whether the image is rough, right, or ruined.

**The knobs on a camera.** One controls focus (steps - sharpness), one controls how literally it follows your instruction (guidance), one is which frame you happened to shoot (seed), and one is a "do not include" list (negative prompt). Same scene, different knobs, very different photo.

The gain: a few dials, each with one job. More is not better - each has a sweet spot.

**📝 `text_to_image.py`** - *diffusers*

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
).to("cuda")

# swap the sampler (scheduler) to trade speed/quality, e.g.:
# from diffusers import DPMSolverMultistepScheduler
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

image = pipe(
    prompt="a red origami crane on a wooden desk, soft window light, photographic",
    negative_prompt="blurry, distorted, extra creases, low quality",  # steer AWAY from these
    num_inference_steps=28,     # sharpness; ~25-30 is the usual range
    guidance_scale=7.0,         # prompt adherence (CFG from 6.1); SDXL sweet spot ~5-8
    width=1024, height=1024,   # SDXL native resolution
    generator=torch.Generator("cuda").manual_seed(42),  # fixed seed = reproducible
).images[0]
image.save("crane.png")
print(image.size)
# Output: (1024, 1024)

- **`num_inference_steps`** - how many denoising passes (Step 4 of 6.1). More means sharper, with diminishing returns past ~30 and wasted latency beyond.

- **`guidance_scale`** - the CFG dial: how hard the image follows the prompt. SDXL likes ~5-8; *SD3.5 and Flux want a lower value (~3.5-4.5)*, which trips people up.

- **`negative_prompt`** - what to steer away from, via the unconditioned pass. A cheap, powerful quality lever ("blurry, extra fingers, watermark").

- **`generator` (seed)** - fixes the random starting noise so a run is reproducible. Change it to get a different image of the same prompt; it changes identity, not quality.

- **sampler (scheduler)** - the algorithm that actually walks the denoising steps (Euler, DPM-Solver, UniPC - the DDIM/DPM-Solver family from Lesson 6.1). A better sampler reaches good quality in fewer steps, so it is the dial you change to buy speed without blindly dropping steps; DPM-Solver is a common default. It is a discrete choice (you pick one), not a slider, which is why the interactive below covers the three continuous dials - steps, guidance, seed - and leaves the sampler here in prose.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Five lines of `diffusers` generate an image, and a handful of dials decide its character. **Steps** control sharpness, **guidance scale** controls prompt adherence (the CFG from 6.1), the **negative prompt** steers away from junk, the **seed** fixes reproducibility, and the **sampler** trades speed for quality. Each has one job and a sweet spot - and those sweet spots differ by model.

---

## 🎯 Concept 3: Image-to-Image and Inpainting - Editing, Not Just Creating

### Image-to-Image and Inpainting - Editing, Not Just Creating

Feed in an existing image, noise it partway, and denoise to a new prompt. Strength sets how much you keep.

**Tracing paper over a drawing.** Lay thin paper down and you still see the original through it (low strength - tiny edits). Lay thick paper and you redraw freely with only a faint guide (high strength - basically new). Strength is how opaque the paper is.

The gain: strength = how much of your image you erase before redrawing. 0 keeps it, 1 replaces it, the middle edits it.

**📝 `img2img_and_inpaint.py`** - *diffusers*

In [ ]:
from diffusers import AutoPipelineForImage2Image, AutoPipelineForInpainting
from diffusers.utils import load_image
import torch

img2img = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# Hosted sample image + matching mask (diffusers inpainting example assets).
_img_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
_mask_url = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo_mask.png"
init = load_image(_img_url).resize((1024, 1024))

# strength controls how much of the input is noised away:
edit = img2img(
    prompt="the same desk at golden hour, warm cinematic light",
    image=init,
    strength=0.45,        # 0 = keep input, 1 = ignore it; 0.3-0.6 edits
    guidance_scale=7.0,
).images[0]
edit.save("edited.png")
print("img2img done")
# Output: img2img done

# INPAINTING: edit ONLY a masked region, copy the rest verbatim.
# Note: inpainting needs a checkpoint fine-tuned for masking - the plain SDXL
# base id won't work here, which is why the model id below is different.
inpaint = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16
).to("cuda")

mask = load_image(_mask_url).resize((1024, 1024))   # white = edit here, black = keep
fixed = inpaint(
    prompt="a ceramic coffee mug", image=init, mask_image=mask, strength=0.85
).images[0]
fixed.save("inpainted.png")
print("inpaint done")
# Output: inpaint done

- `AutoPipelineForImage2Image` reuses the same checkpoint - it just starts denoising from your noised input instead of pure noise.

- **`strength` is the whole game:** it sets how much noise is added to the input before denoising. Low keeps your image (touch-ups), ~0.45 edits it (keeps layout, changes look), near 1.0 regenerates from scratch.

- **Inpainting** uses a mask-fine-tuned checkpoint (hence the different model id) and adds a `mask_image`: white pixels are regenerated to the new prompt, black pixels are copied from the input verbatim. We use a high `strength` (~0.85) here on purpose - the mask, not strength, protects the rest, so we want a near-full redraw *inside* the masked region. This is how you swap a product, fix a hand, or remove an object without touching the rest.

- `AutoPipelineFor*` picks the right pipeline class for the checkpoint, so the same code works across SDXL, SD3.5, and Flux with only the model id changed.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Image-to-image noises your input partway and denoises it toward a new prompt; **strength** sets how far it noises, so low keeps your image, mid edits it, and high regenerates it. **Inpainting** adds a mask so only the masked region changes and the rest is copied verbatim. Same pipeline as text-to-image - it just starts from your image instead of pure noise.

---

## 🎯 Concept 4: The Model Zoo - SD1.5 to Flux.2

### The Model Zoo - SD1.5 to Flux.2

Which base to start from is a real decision - it sets your quality, your ecosystem, and your license.

**Choosing a camera system, not just a camera body.** You do not only pick the sharpest body - you pick the one with the lenses you need, that fits your budget, and that you are licensed to use commercially. A slightly older body with a huge lens lineup often beats the newest body with none.

The gain: base model = a whole system (quality + ecosystem + license). Newest is not automatically best for your job.

| Base model | Backbone | Strengths | License note |
|---|---|---|---|
| SD 1.5(2022) | U-Net, 512px | Tiny, runs anywhere; legacy LoRA library | Open (CreativeML) |
| SDXL(2023) | U-Net, 1024px, dual CLIP | Huge free LoRA/checkpoint ecosystem; no revenue cap | Open, commercial-friendly |
| SD 3.5(2024) | MMDiT, rectified flow | Strong prompt following; Stability community license | Free under revenue threshold |
| Flux.1(2024) | DiT, flow matching | Big quality jump; Schnell is Apache-2.0 | Dev = non-commercial; Schnell = Apache-2.0 |
| Flux.2(Nov 2025) | DiT, up to 4MP | Best quality, text rendering, multi-reference | Klein 4B = Apache-2.0; larger = paid commercial |

#### 💡 What just happened

⚡ What just happened?
  The open ecosystem is a **zoo of base models**: SD1.5 (tiny, legacy), SDXL (the commercial-friendly workhorse with the biggest free LoRA ecosystem), SD3.5 (strong rectified-flow line), and Flux.1/Flux.2 (the 2026 quality leaders). Choosing one balances quality, ecosystem size, VRAM, and license. **Civitai** is the community hub where checkpoints and LoRAs are shared.

---

## 🎯 Concept 5: Customize It - LoRA and Fine-Tuning

### Customize It - LoRA and Fine-Tuning

Teach a base model your style, character, or product with a file smaller than a photo.

**A clip-on lens filter.** You do not buy a new camera to get a warm-tone look - you clip a small filter onto the lens you have. A LoRA is that filter for a base model: tiny, snap-on, and you can swap or stack them.

The gain: LoRA = a small adapter clipped onto a frozen base. The base never changes; the adapter carries the style.

**📝 `use_lora.py`** - *diffusers*

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# Load a LoRA adapter (a 4-20 MB .safetensors file - the standard safe-to-load
# weight format - e.g. from Civitai). Name it so we can reference it below.
pipe.load_lora_weights("TheLastBen/Papercut_SDXL", weight_name="papercut.safetensors",
                       adapter_name="cutout")
pipe.set_adapters(["cutout"], adapter_weights=[0.8])   # dial the strength

# Stack a second LoRA (e.g. a character) by naming each and listing both:
# pipe.load_lora_weights("./loras", weight_name="my-character.safetensors",
#                        adapter_name="char")
# pipe.set_adapters(["cutout", "char"], adapter_weights=[0.8, 0.7])

image = pipe(
    prompt="a fox in a forest, paper-cutout style",  # trigger word from the LoRA
    num_inference_steps=28, guidance_scale=7.0,
).images[0]
image.save("fox.png")
print("LoRA applied - base weights never changed")
# Output: LoRA applied - base weights never changed

- `load_lora_weights` attaches the adapter to the frozen base pipeline - the multi-GB base is untouched, the few-MB adapter does the restyling.

- `set_adapters([...], adapter_weights=[0.8])` activates the adapter (referenced by the `adapter_name` you gave it on load) and dials how strongly it applies. Name a second LoRA too and list both to **stack them** (a style + a character), as the commented lines show.

- Most LoRAs have a **trigger word** (here "paper-cutout style") you include in the prompt to activate the learned concept.

- To *train* one, you use DreamBooth+LoRA on ~5-20 images: rank 16-32 (rank = how much the adapter is allowed to learn - higher rank means more capacity but a larger file), learning rate ~1e-4, 1024px for SDXL (both text encoders). That is the same low-rank adaptation you will meet again for LLMs in Module 9, where we go into the matrix math.

> 📦 **Info**
>
> 🎨 LoRA vs DreamBooth vs full fine-tune - when to use which
> - **LoRA** - a tiny adapter (4-20 MB) for a style, character, or product. The default for almost everything; trains on one consumer GPU, stacks, ships easily.
> - **DreamBooth** - the *training recipe* (5-20 images of one subject, a unique token) usually combined with LoRA. Reach for it to nail a specific person or product.
> - **Full fine-tune** - retrain all the base weights. Only for a large, distinctive dataset and a serious compute budget; produces a multi-GB checkpoint, not a portable adapter.

#### 💡 What just happened

⚡ What just happened?
**LoRA** freezes the base model and trains tiny low-rank adapters in the denoiser's attention layers, so a 4-20 MB file can restyle a multi-GB model, stack with other adapters, and ship anywhere. **DreamBooth** is the recipe for teaching a specific subject from a handful of images; a **full fine-tune** is the heavyweight option for large, distinctive datasets. Most real customization is LoRA.

---

## 🎯 Concept 6: Ship It - Tooling, Speed, and Production

### Ship It - Tooling, Speed, and Production

From a notebook to a service: ComfyUI workflows, few-step speedups, serverless GPUs, and safety.

**From home cooking to a restaurant line.** Cooking one dish in your kitchen (a notebook) is fine; serving a full restaurant needs a repeatable line (a workflow), faster techniques (few-step models), and a kitchen you rent only when busy (serverless GPU). Same recipe, industrial setup.

The gain: production = repeatable workflow + fewer steps + rented GPU + a safety check. The image quality is the easy part; shipping it cheaply is the work.

**📝 `production_notes.py`** - *diffusers*

In [ ]:
from diffusers import StableDiffusionXLPipeline, LCMScheduler
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")

# SPEED: an LCM-LoRA turns any SDXL into a ~4-step model.
pipe.load_lora_weights("latent-consistency/lcm-lora-sdxl")  # a single LoRA is active on load - no set_adapters needed
pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)

image = pipe(
    prompt="a ceramic mug on a marble counter, product photo",
    num_inference_steps=4,      # 4 instead of 28 -> ~7x less compute
    guidance_scale=1.5,         # distilled models need very low guidance (see note)
).images[0]

# SAFETY: never ship without an output check (deepfake / NSFW / brand risk).
def passes_safety(img) -> bool:
    # plug in your classifier / provider safety checker here
    return True

print("ok" if passes_safety(image) else "blocked")
# Output: ok

- **Few-step generation** - an LCM-LoRA (LCM = Latent Consistency Model, a distillation that lets a model reach a good image in ~4 big jumps instead of ~28 small denoising steps; also SDXL Lightning, Flux Schnell) drops a ~28-step render to ~4 steps, the single biggest speed/cost win. These distilled models need a *very* low `guidance_scale` (~1-2) **regardless of the base model** - that is a property of the distillation, not a contradiction of Step 2's SDXL sweet spot (~5-8).

- **Quantization** - storing the model's weights in fewer bits (FP8 or NF4 instead of the usual 16-bit), which shrinks its memory footprint so it fits on a smaller, cheaper GPU. NVIDIA reported roughly 40% less VRAM for Flux.2 this way.

- **ComfyUI** - the de-facto production environment: node-graph workflows with a REST + WebSocket API, day-one support for new models, and one-click deploy services. It is how teams turn a pipeline into a reproducible, automatable service.

- **Deployment + safety** - serverless GPU (Modal, RunPod, fal.ai, Replicate) scales to zero between requests, usually beating a self-hosted GPU until utilization is high. Always gate outputs with a safety checker; deeper cost work is Modules 12-13, and responsible generation is Module 15.

#### 💡 What just happened

⚡ What just happened?
  Shipping is its own skill. **Few-step models** (LCM-LoRA, Lightning, Flux Schnell) cut ~28 steps to ~4 - the biggest cost lever - and **quantization** (FP8/NF4) shrinks VRAM. **ComfyUI** is the production environment that turns a pipeline into an automatable, API-driven service, and **serverless GPU** scales to zero between requests. Always gate outputs with a safety checker.

## Synthesis: ship a brand catalog through the whole ecosystem

### The complete picture, in one breath

Say you need on-brand product images at scale. **Assemble** the pipeline - text encoder, denoiser, VAE - and pick a **base** by quality and license (SDXL for its free ecosystem and clean commercial terms, or Flux.2 if you can license it). **Generate** with the right dials (modest steps, the model's guidance sweet spot, a negative prompt). **Edit** existing shots with img2img and swap details with inpainting. **Customize** to the brand look with a stacked **LoRA**. Then **ship** it: a ComfyUI workflow, a few-step LCM model for cost, serverless GPU for scale, and a safety checker on every output. That is the Stable Diffusion ecosystem - an open pipeline you operate end to end.

> ℹ️ **Info**
>
> Where this goes next in Module 6
> - **Lesson 6.3 - Controlled Generation:** ControlNet and IP-Adapter - impose exact pose, depth, and edges, the structural control we deferred here.
> - **Lesson 6.4 - ViT & CLIP:** inside the text encoder - how CLIP aligns text and images in one embedding space.
> - **Lesson 6.6 - Video & 3D Generation:** the same diffusion ecosystem extended across time and space.
> - **Modules 12-13 and 15:** production deployment and cost in depth, and responsible image generation.

- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (2022) - [arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)

- Podell et al., *SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis* (2023) - [arxiv.org/abs/2307.01952](https://arxiv.org/abs/2307.01952)

- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021) - [arxiv.org/abs/2106.09685](https://arxiv.org/abs/2106.09685)

- Ruiz et al., *DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation* (2022) - [arxiv.org/abs/2208.12242](https://arxiv.org/abs/2208.12242)

- Hugging Face *diffusers* library and LoRA/DreamBooth training guide - [huggingface.co/docs/diffusers](https://huggingface.co/docs/diffusers)

- ComfyUI - [github.com/comfyanonymous/ComfyUI](https://github.com/comfyanonymous/ComfyUI) · Civitai community hub - [civitai.com](https://civitai.com)

## Recap

> ✅ **Info**
>
> What we covered
> - **The pipeline** - text encoder + denoiser + VAE + scheduler; three swappable trained parts.
> - **Text-to-image dials** - steps, guidance scale, sampler, seed, negative prompt; each with a model-specific sweet spot.
> - **img2img and inpainting** - strength sets how much you keep; masks edit a region and copy the rest.
> - **The model zoo** - SD1.5 / SDXL / SD3.5 / Flux.1 / Flux.2; choose by quality, ecosystem, VRAM, and license.
> - **LoRA and fine-tuning** - tiny frozen-base adapters; DreamBooth for subjects; full fine-tune for big datasets.
> - **Production** - ComfyUI, few-step/quantization for speed, serverless GPU for scale, safety on every output.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Stable Diffusion Ecosystem- From Understanding to Shipping**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_2.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.2.html` - regenerate this notebook whenever the source HTML is updated.*
